<a href="https://colab.research.google.com/github/MLWithMathematics/Fly_Rank_ML-Intern/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MLWithMathematics/Fly_Rank_ML-Intern/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Clustering (unsupervised). The question is "what performance archetypes exist across the content inventory?" — not "predict a label" or "rank items." I'm looking for natural groupings in the multivariate space of search performance, engagement, content properties, and freshness signals. The output is a cluster assignment per page, with each cluster profiled and named by its typical characteristics, then mapped to a recommended action.

Why not classification? I have no ground-truth archetype labels — the archetypes are the discovery. Why not ranking? I'm not ordering pages by a single priority score; I'm segmenting them into qualitatively different types that each warrant a different treatment.

In [6]:
# verify this is a clustering-friendly shape
import pandas as pd
import os
LOCAL_PATH = "data/raw/content_refresh_anonymized.csv"
RAW_URL = "https://raw.githubusercontent.com/MLWithMathematics/Fly_Rank_ML-Intern/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(LOCAL_PATH if os.path.exists(LOCAL_PATH) else RAW_URL)
print(f"Shape: {df.shape}")
print(f"Rows: {df.shape[0]:,}, Columns: {df.shape[1]}")
print(f"Clients: {df['client_id'].nunique()}")
print(f"\nNumeric columns available for clustering:")
numeric_cols = df.select_dtypes(include='number').columns.tolist()
print(f"  {len(numeric_cols)} numeric features")
print(f"\nCategorical columns available:")
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f"  {len(cat_cols)} categorical features")

Shape: (30000, 44)
Rows: 30,000, Columns: 44
Clients: 32

Numeric columns available for clustering:
  30 numeric features

Categorical columns available:
  14 categorical features


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

There is no target. This is unsupervised — I am not predicting a label. The cluster assignment itself is the output, discovered from the data's structure.

In [7]:
# confirm no target, show feature/exclusion split
CLUSTER_FEATURES = [
    'search_volume', 'competition', 'cpc', 'word_count',
    'content_age_days', 'days_since_last_update',
    'impressions_90d', 'clicks_90d', 'sessions_90d',
    'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d',
    'days_with_impressions', 'days_with_sessions',
    'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct'
]

EXCLUDED_LEAKAGE = ['trend_pct', 'trend_direction', 'is_declining_label',
                     'impressions_last_30d', 'impressions_prev_30d',
                     'clicks_last_30d', 'clicks_prev_30d',
                     'sessions_last_30d', 'sessions_prev_30d']

EXCLUDED_IDS = ['content_id', 'client_id']

print("Features for clustering:", len(CLUSTER_FEATURES))
print("Excluded (leakage):", EXCLUDED_LEAKAGE)
print("Excluded (IDs):", EXCLUDED_IDS)

# Verify none of the excluded columns snuck in
assert not set(EXCLUDED_LEAKAGE) & set(CLUSTER_FEATURES), "Leakage!"
assert not set(EXCLUDED_IDS) & set(CLUSTER_FEATURES), "ID leak!"
print("\n✓ No leakage columns in feature set")

Features for clustering: 19
Excluded (leakage): ['trend_pct', 'trend_direction', 'is_declining_label', 'impressions_last_30d', 'impressions_prev_30d', 'clicks_last_30d', 'clicks_prev_30d', 'sessions_last_30d', 'sessions_prev_30d']
Excluded (IDs): ['content_id', 'client_id']

✓ No leakage columns in feature set


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Human sense-check — inspect the top-10 pages from each cluster and verify the archetype name makes intuitive sense. If a cluster called "champions" contains pages with 5 impressions and no clicks, the clustering is wrong regardless of the silhouette score.



In [8]:
 # demonstrate the metric is computable today
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np

# Quick feasibility: scale features and try K=5
feature_df = df[CLUSTER_FEATURES].copy()
# Handle missingness: fill NaN with 0 + add has_ flags for key columns
for col in feature_df.columns:
    if feature_df[col].isna().any():
        feature_df[f'has_{col}'] = (~feature_df[col].isna()).astype(int)
        feature_df[col] = feature_df[col].fillna(0)

# Log-transform heavy-tailed columns
for col in ['impressions_90d', 'clicks_90d', 'sessions_90d',
            'ai_sessions_90d', 'scroll_events_90d', 'search_volume']:
    if col in feature_df.columns:
        feature_df[col] = np.log1p(feature_df[col])

scaler = StandardScaler()
X = scaler.fit_transform(feature_df)

km = KMeans(n_clusters=5, random_state=42, n_init=10)
labels = km.fit_predict(X)

sil = silhouette_score(X, labels)
print(f"Quick K=5 silhouette score: {sil:.3f}")
print(f"Cluster sizes: {np.bincount(labels)}")

Quick K=5 silhouette score: 0.179
Cluster sizes: [9906 6749 6944 2464 3937]


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [9]:

print(f"Total rows: {df.shape[0]:,}")
print(f"Unique content_id: {df['content_id'].nunique():,}")
print(f"Grain is 1:1: {df.shape[0] == df['content_id'].nunique()}")
print(f"Clients: {df['client_id'].nunique()}")
print()
print("Sample — one row = one page:")
display_cols = ['content_id', 'client_id', 'impressions_90d', 'clicks_90d',
                'ctr', 'avg_position', 'engagement_rate', 'content_age_days',
                'word_count', 'content_type']
print(df[display_cols].head(3).to_string(index=False))

Total rows: 30,000
Unique content_id: 30,000
Grain is 1:1: True
Clients: 32

Sample — one row = one page:
          content_id         client_id  impressions_90d  clicks_90d  ctr  avg_position  engagement_rate  content_age_days  word_count    content_type
content_304f48230142 client_f369cb89fc             3803          29 0.76          10.6             5.88               187      3221.0 keyword article
content_a1fb4e703a9e client_4e07408562            15320           7 0.05          20.3             0.00               445      2481.0 keyword article
content_9aa793d4d895 client_7f2253d7e2            12581          11 0.09          36.5             0.00               141      3515.0 keyword article


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

The starter pipeline already uses a hand-written baseline score with 4 weighted components and 6 reason-code rules. That rule works — but it makes three simplifications that clustering overcomes:

It assumes one priority axis. The baseline produces a single score: higher = fix first. But a page that's "high impressions, good position, terrible engagement" is a fundamentally different problem from a page that's "low impressions, old content, no clicks." They need different actions (engagement fix vs. content rewrite), not just different scores. Clustering discovers these qualitatively different types.

The feature interactions are too many for manual thresholds. With 15+ features, the number of meaningful threshold combinations is combinatorial. The baseline uses 6 hand-tuned rules. Clustering lets the data's own geometry decide where the natural group boundaries fall — without a human pre-deciding every cutoff.

Fixed rules don't adapt to the data's shape. The same thresholds applied to 32 different clients may carve very different proportions. A client whose median impressions are 50 looks nothing like one whose median is 50,000. Clustering adjusts to the actual distribution of the data it sees.

The evidence from the starter pipeline: the baseline's Precision@50 is 0.240. The random forest achieves 0.740 — a ~3x lift. This proves that the signals contain learnable structure beyond what fixed rules capture. Clustering captures that same structure in an unsupervised way, without needing labels.

In [10]:


# Baseline-style binary flags (from the starter pipeline)
df['flag_stale_visible'] = ((df['days_since_last_update'] >= 180) &
                             (df['impressions_90d'] >= 500)).astype(int)
df['flag_thin_visible'] = ((df['word_count'] > 0) & (df['word_count'] < 1200) &
                            (df['impressions_90d'] >= 250)).astype(int)

# How many pages get NO flag at all?
no_flag = ((df['flag_stale_visible'] == 0) & (df['flag_thin_visible'] == 0)).sum()
print(f"Pages with no baseline flag: {no_flag:,} / {len(df):,} ({no_flag/len(df)*100:.1f}%)")
print("→ The fixed rules leave the majority of pages un-typed.")
print()

# But K-Means assigns every page to a cluster
print(f"Pages with a cluster assignment: {len(labels):,} / {len(df):,} (100%)")
print("→ Clustering types the full inventory, not just the obvious cases.")

Pages with no baseline flag: 29,901 / 30,000 (99.7%)
→ The fixed rules leave the majority of pages un-typed.

Pages with a cluster assignment: 30,000 / 30,000 (100%)
→ Clustering types the full inventory, not just the obvious cases.


## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅ ] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.